In [ ]:
import os, re, json, warnings
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import spacy

warnings.filterwarnings('ignore')
BASE_DIR = r''
RAW_PATH = os.path.join(BASE_DIR, 'Data', 'dataset_ftp_04032026.json')
CLEAN_PATH = os.path.join(BASE_DIR, 'Data', 'dataset_clean_cnj.json')
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')
FIG_DIR = os.path.join(BASE_DIR, 'Results', 'Figures')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('Loading data...')
with open(RAW_PATH, 'r', encoding='utf-8', errors='replace') as f:
    df = pd.DataFrame(json.load(f))
FIELDS = ['inteiro_teor', 'fato', 'direito', 'pedido']

ANON_STATS = defaultdict(int)

## Section 1: Regex Engine with Tracking


In [ ]:
EXTREME_REGEX_PII = [
    (r'\b\d{3}\.\d{3}\.\d{3}-\d{2}\b', '[CPF]'),
    (r'\b\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}\b', '[CNPJ]'),
    (r'\b\d{1,2}\.?\d{3}\.?\d{3}-?[a-zA-Z0-9]{1,2}\b', '[RG_DOCUMENTO]'),
    (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,7}\b', '[EMAIL]'),
    (r'\(?\d{2}\)?[\s.-]?\d{4,5}[\s.-]?\d{4}\b', '[TELEFONE]'),
    (r'\b[A-Z]{3}-?\d[A-Z0-9]\d{2}\b', '[PLACA_VEICULO]'),
    (r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', '[ENDERECO_IP]'),
    (r'\b\d{5}-\d{3}\b', '[CEP]'),
    (r'\b\d{11}\b', '[DOC_11_DIGITOS]'),
    (r'\b\d{7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}\b', '[PROCESSO_CNJ]'),
    (r'\bOAB/?[A-Z]{2}\s*n[°o\.]*\s*\d{3,6}\b', '[OAB]'),
    (r'(?i)Processo\s*(?:n[°o\.]*\s*)?\d+', '[PROCESSO]'),
    (r'(?i)Inquérito\s*(?:n[°o\.]*\s*)?\d+', '[INQUERITO]'),
    (r'R\$\s*\d{1,3}(?:\.\d{3})*,\d{2}', '[VALOR_MONETARIO]'),
    (r'\b\d{1,2}/\d{1,2}/\d{2,4}\b', '[DATA]'),
    (r'\b\d{1,3}\s*(?:anos|meses)\s*de\s*idade\b', '[IDADE_OCULTA]'),
    (r'\b(?:conta|agência)\s*\d+[-.]?\d*\b', '[DADO_BANCARIO]'),
]

def apply_regex_pii(text):
    if not isinstance(text, str): return str(text)
    for pat, rep in EXTREME_REGEX_PII:
        def track_replace(match):
            ANON_STATS[rep] += 1
            return rep
        text = re.sub(pat, track_replace, text, flags=re.IGNORECASE)
    return text

## Section 2: Hyper-Sensitive Dictionary Engine


In [ ]:
HYPER_SENSITIVE_DICTS = {
    '[TRIBUNAL]': ['tribunal de justiça', 'stf', 'stj', 'tst', 'tse', 'trf', 'tjgo', 'tjsp', 'tjmt', 'tjmg'],
    '[VARA_COMARCA]': ['vara cível', 'vara criminal', 'vara de família', 'comarca de'],
    '[ATOR_JURIDICO]': ['juiz', 'juíza', 'desembargador', 'promotor', 'vossa excelência', 'perito', 'oficial de justiça', 'delegado'],
    '[PAPEL_PROCESSUAL]': ['parte autora', 'requerente', 'exequente', 'parte ré', 'requerido', 'executado', 'testemunha', 'informante', 'vítima', 'ofendida'],
    '[VIOLENCIA_ABUSO]': ['abuso sexual', 'violência doméstica', 'violência contra mulher', 'estupro', 'assédio sexual', 'agressão física', 'medida protetiva'],
    '[PSIQUIATRIA_SUICIDIO]': ['suicídio', 'ideação suicida', 'autolesão', 'histórico psiquiátrico', 'internação compulsória', 'dependência química', 'vício em drogas'],
    '[FAMILIA_CRITICO]': ['paternidade', 'filiação biológica', 'adoção', 'infertilidade', 'aborto', 'conselho tutelar', 'abrigo'],
    '[SAUDE_GERAL]': ['prontuário', 'laudo médico', 'hiv', 'aids', 'ist', 'câncer', 'oncologista', 'depressão', 'esquizofrenia', 'doença crônica', 'cid'],
    '[BIOMETRIA_GENETICA]': ['biometria', 'impressão digital', 'reconhecimento facial', 'dna', 'código genético', 'exame genético'],
    '[RELIGIAO_RACA]': ['evangélico', 'católico', 'espírita', 'umbanda', 'candomblé', 'igreja', 'terreiro', 'negro', 'pardo', 'índio', 'indígena', 'branco'],
    '[POLITICA_SINDICATO]': ['vereador', 'prefeito', 'petista', 'bolsonarista', 'sindicato', 'sindicalista', 'partido político'],
    '[VIDA_SEXUAL]': ['homossexual', 'heterossexual', 'bissexual', 'transgênero', 'orientação sexual']
}

COMPILED_HYPER = []
for tok, words in HYPER_SENSITIVE_DICTS.items():
    pattern = r'\b(' + '|'.join([re.escape(w) for w in words]) + r')\b'
    COMPILED_HYPER.append((re.compile(pattern, re.IGNORECASE), tok))

def apply_hyper_sensitive(text):
    for rgx, tok in COMPILED_HYPER:
        def dict_replace(match):
            ANON_STATS[tok] += 1
            return tok
        text = rgx.sub(dict_replace, text)
    return text

## Section 3: NLP Inference (Spacy)


In [ ]:
SPACY_CANDIDATES = ['pt_core_news_md', 'pt_core_news_sm']
nlp = None
loaded_model_name = None

for model_name in SPACY_CANDIDATES:
    try:
        nlp = spacy.load(model_name, disable=['parser', 'tagger', 'lemmatizer', 'attribute_ruler'])
        loaded_model_name = model_name
        break
    except Exception:
        continue

if nlp is None:
    raise RuntimeError(
        'Nenhum modelo spaCy PT encontrado. Instale um destes:\n'
        'python -m spacy download pt_core_news_md\n'
        'python -m spacy download pt_core_news_sm'
    )

nlp.max_length = 20_000_000
print(f'spaCy model loaded: {loaded_model_name}')

def mask_spacy_entities(text):
    text = str(text)
    doc = nlp(text)
    masked = text

    for ent in sorted(doc.ents, key=lambda x: x.start_char, reverse=True):
        if ent.label_ == 'PER':
            masked = masked[:ent.start_char] + '[PESSOA_FISICA]' + masked[ent.end_char:]
            ANON_STATS['[PESSOA_FISICA_NLP]'] += 1
        elif ent.label_ == 'ORG':
            masked = masked[:ent.start_char] + '[INSTITUICAO]' + masked[ent.end_char:]
            ANON_STATS['[INSTITUICAO_NLP]'] += 1
        elif ent.label_ in ('LOC', 'GPE'):
            masked = masked[:ent.start_char] + '[LOCAL_NLP]' + masked[ent.end_char:]
            ANON_STATS['[LOCAL_NLP]'] += 1

    return masked

## Section 4: Applying the Pipeline


In [ ]:
tqdm.pandas(desc='Applying CNJ Sandbox')
for f in FIELDS:
    print(f'\nProcessing field: {f}...')
    df[f] = df[f].apply(apply_regex_pii)
    df[f] = df[f].apply(apply_hyper_sensitive)
    df[f] = df[f].progress_apply(mask_spacy_entities)
    df[f] = df[f].str.replace(r'\s+', ' ', regex=True).str.strip()

df.to_json(CLEAN_PATH, orient='records', force_ascii=False, indent=2)
print(f'\nCNJ Bulletproof Anonymization Complete. Saved to {CLEAN_PATH}')

## Section 5: Tracking & Results Plotting


In [ ]:
with open(os.path.join(RESULTS_DIR, 'anonymization_stats_cnj.json'), 'w') as f:
    json.dump(ANON_STATS, f, indent=2, ensure_ascii=False)

top_stats = dict(sorted(ANON_STATS.items(), key=lambda item: item[1], reverse=True)[:20])

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(list(top_stats.keys()), list(top_stats.values()), color='#2980B9', alpha=0.9)
ax.invert_yaxis()
for bar in bars:
    width = bar.get_width()
    ax.text(width + (max(top_stats.values()) * 0.02), bar.get_y() + bar.get_height()/2, 
            f'{int(width):,}', va='center', weight='bold', fontsize=9)

ax.set_title('Top 20 Protected Information Classes Masked', weight='bold', pad=15)
ax.set_xlabel('Number of Replacements Made')
ax.spines['right'].set_visible(False) 
ax.spines['top'].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.4)

plot_path = os.path.join(FIG_DIR, 'preprocessing', 'fig_anonymization_distribution.png')
os.makedirs(os.path.dirname(plot_path), exist_ok=True)
plt.tight_layout()
plt.savefig(plot_path, dpi=300)
plt.show()
print(f'Tracking plot saved to {plot_path}')

In [ ]:
AUDIT_PATH_JSON = os.path.join(RESULTS_DIR, 'anonymization_audit_summary.json')
AUDIT_PATH_CSV  = os.path.join(RESULTS_DIR, 'anonymization_audit_summary.csv')

audit_rows = []
for f in FIELDS:
    total_chars = df[f].astype(str).str.len().sum()
    total_docs  = len(df)
    token_hits  = sum(v for k, v in ANON_STATS.items())
    audit_rows.append({
        'field': f,
        'documents': total_docs,
        'total_chars_after_masking': int(total_chars),
        'spacy_model_used': loaded_model_name,
        'global_mask_replacements': int(token_hits),
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(AUDIT_PATH_CSV, index=False, encoding='utf-8-sig')

with open(AUDIT_PATH_JSON, 'w', encoding='utf-8') as f:
    json.dump({
        'spacy_model_used': loaded_model_name,
        'total_documents': int(len(df)),
        'mask_stats': dict(ANON_STATS),
        'per_field_rows': audit_rows,
    }, f, indent=2, ensure_ascii=False)

print('\n=== ANONYMIZATION AUDIT SUMMARY ===')
print(audit_df.to_string(index=False))
print(f'\nSaved CSV  -> {AUDIT_PATH_CSV}')
print(f'Saved JSON -> {AUDIT_PATH_JSON}')